# Module 02 — Explorer l'ingest RSS dans un notebook

**Objectif** : comprendre *ligne par ligne* ce que font `feeds.py`, `poll.py`, `seen.py` et `cli.py` avant de les finaliser dans `src/presslake/`.

**Prérequis**
- Être à la racine du repo (`data_project/`)
- `uv sync` déjà exécuté (`feedparser`, `httpx`, `pyyaml`)
- `config/feeds.yml` rempli

**Lancer le notebook** (si besoin) :
```bash
uv add --dev ipykernel jupyter
uv run jupyter notebook notebooks/02-rss-ingest-exploration.ipynb
```

Ou ouvre ce fichier directement dans Cursor / VS Code (mode notebook).

## Carte des fichiers — qui fait quoi ?

```
config/feeds.yml          → liste des flux (CONFIG, pas du code)
        │
        ▼
feeds.py                  → lit le YAML → objets Feed
        │
        ▼
poll.py                   → httpx (HTTP) + feedparser (XML) + item_key
        │
        ▼
seen.py                   → mémorise les clés déjà vues (data/seen.json)
        │
        ▼
cli.py                    → `presslake poll` assemble tout
```

Dans ce notebook, on refait **la même chaîne** cellule par cellule, sans importer encore ton package (tu comprends d'abord, tu factorises ensuite).

## Étape 0 — Imports et chemins

On importe les bibliothèques et on fixe le répertoire de travail sur la racine du projet.

In [13]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import feedparser  # parse RSS / Atom → dict Python
import httpx       # client HTTP moderne (remplace requests)
import yaml        # lit config/feeds.yml

# Racine du repo : le notebook est dans notebooks/, on remonte d'un cran
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FEEDS_PATH = ROOT / "config" / "feeds.yml"   # équivalent défaut de feeds.py
SEEN_PATH = ROOT / "data" / "seen.json"      # équivalent défaut de seen.py
USER_AGENT = "PressLake/0.1 (learning; local dev)"  # obligatoire pour France24, RFI, CNIL

print("Racine projet :", ROOT)
print("Fichier flux  :", FEEDS_PATH, "→ existe:", FEEDS_PATH.exists())

Racine projet : /home/anthony-marais/Documents/data_project
Fichier flux  : /home/anthony-marais/Documents/data_project/config/feeds.yml → existe: True


## Étape 1 — `feeds.py` : modèle `Feed` + chargement YAML

**Rôle** : transformer un fichier config en objets Python typés.

- `@dataclass(frozen=True)` → objet immuable (on ne modifie pas un feed en vol)
- `yaml.safe_load` → parse le YAML en dict/list Python
- `Feed(**row)` → crée un Feed à partir de chaque entrée sous `feeds:`

In [14]:
@dataclass(frozen=True)
class Feed:
  id: str        # identifiant court, stable (ex: france24) — sert à la clé de dédup
  name: str      # libellé humain
  url: str       # URL du flux RSS/Atom
  lang: str      # fr, en…
  category: str  # presse, institution, tech…


def load_feeds(path: Path | None = None) -> list[Feed]:
  """Équivalent de src/presslake/ingest/feeds.py"""
  path = path or FEEDS_PATH
  # read_text lit tout le fichier en str ; encoding=utf-8 gère les accents (Le Monde, RFI…)
  raw = yaml.safe_load(path.read_text(encoding="utf-8"))
  # raw["feeds"] est une liste de dicts ; Feed(**row) mappe les clés YAML → attributs
  return [Feed(**row) for row in raw["feeds"]]


feeds = load_feeds()
print(f"{len(feeds)} flux chargés")
for f in feeds:
  print(f"  - {f.id:14} {f.category:12} {f.url}")

7 flux chargés
  - france24       presse       https://www.france24.com/fr/rss
  - rfi            presse       https://www.rfi.fr/fr/rss
  - lemonde-une    presse       https://www.lemonde.fr/rss/une.xml
  - bbc-world      presse       https://feeds.bbci.co.uk/news/world/rss.xml
  - cnil           institution  https://www.cnil.fr/fr/rss.xml
  - eu-press       institution  https://ec.europa.eu/commission/presscorner/api/rss
  - hnrss          tech         https://hnrss.org/frontpage


## Étape 2 — Regarder le YAML brut

Utile pour vérifier que la structure correspond à ce que `load_feeds` attend.

In [15]:
raw_yaml = yaml.safe_load(FEEDS_PATH.read_text(encoding="utf-8"))
print("Clés racine :", list(raw_yaml.keys()))  # attendu : ['feeds']
print("Premier feed (dict) :", raw_yaml["feeds"][0])

Clés racine : ['feeds']
Premier feed (dict) : {'id': 'france24', 'name': 'France 24', 'url': 'https://www.france24.com/fr/rss', 'lang': 'fr', 'category': 'presse'}


## Étape 3 — `poll.py` : fetch HTTP (couche réseau)

**Rôle** : télécharger le XML du flux.

| Paramètre | Pourquoi |
|---|---|
| `headers={"User-Agent": ...}` | Sans ça → 403 sur plusieurs médias FR |
| `timeout=30.0` | Évite de bloquer si le serveur ne répond pas |
| `follow_redirects=True` | Certains flux redirigent (308, 301) |
| `raise_for_status()` | Lève une erreur si HTTP 4xx/5xx |

⚠️ **Piège fréquent** : le paramètre s'appelle `headers` (avec un **s**), pas `header`.

In [16]:
def fetch_feed(url: str) -> feedparser.FeedParserDict:
  """Équivalent de la partie réseau de poll.py"""
  response = httpx.get(
    url,
    headers={"User-Agent": USER_AGENT},  # ← headers (pluriel)
    timeout=30.0,
    follow_redirects=True,
  )
  response.raise_for_status()  # plante si 403, 404, 500…
  # feedparser attend du texte XML, pas des bytes bruts
  return feedparser.parse(response.text)


# Test sur UN seul flux pour isoler les problèmes
test_feed = feeds[0]  # france24
parsed = fetch_feed(test_feed.url)

print("Titre du flux :", parsed.feed.get("title"))
print("Nombre d'items :", len(parsed.entries))
print("Type du 1er item :", type(parsed.entries[0]))

Titre du flux : France 24 - Infos, news & actualités - L'information internationale en direct
Nombre d'items : 24
Type du 1er item : <class 'feedparser.util.FeedParserDict'>


## Étape 4 — Inspecter un item RSS

`feedparser` convertit chaque `<item>` en objet style dict. Les champs utiles pour PressLake :

In [17]:
entry = parsed.entries[0]  # premier article du flux

# .get() comme un dict — certains champs peuvent manquer selon le flux
print("title :", entry.get("title"))
print("link  :", entry.get("link"))
print("id    :", entry.get("id"))      # souvent présent en Atom
print("guid  :", entry.get("guid"))    # souvent présent en RSS 2.0
print("published :", entry.get("published"))

# Toutes les clés disponibles sur CET item (varie selon le flux)
print("\nClés de l'entry :", list(entry.keys())[:15], "...")

title : Présidentielle 2027 : pourquoi Marine Le Pen bénéficie d'un électorat plus stable que ses rivaux
link  : https://www.france24.com/fr/france/20260831-pr%C3%A9sidentielle-2027-pourquoi-marine-le-pen-b%C3%A9n%C3%A9ficie-%C3%A9lectorat-plus-stable-rivaux
id    : 24ff1a0e-a52a-11f1-85a0-3d03a70b1472
guid  : 24ff1a0e-a52a-11f1-85a0-3d03a70b1472
published : Mon, 31 Aug 2026 12:38:16 GMT

Clés de l'entry : ['tags', 'title', 'title_detail', 'links', 'link', 'summary', 'summary_detail', 'media_thumbnail', 'href', 'id', 'guidislink', 'published', 'published_parsed', 'source', 'authors'] ...


## Étape 5 — `item_key()` : clé stable pour la dédup

**Rôle** : une seule fonction qui choisit la meilleure clé, dans l'ordre :

1. `id` (Atom)
2. `guid` (RSS)
3. `link` (fallback universel)

Si aucun des trois → on lève une erreur (flux à corriger ou item à ignorer).

In [18]:
def item_key(entry: dict) -> str:
  """Équivalent de poll.py — clé stable par article"""
  for field in ("id", "guid", "link"):
    value = entry.get(field)
    if value:
      return str(value).strip()  # str() au cas où guid serait un objet feedparser
  raise ValueError(f"item sans clé : title={entry.get('title')!r}")


# Clé composite : feed_id + clé item → évite collision entre deux flux
def composite_key(feed: Feed, entry: dict) -> str:
  return f"{feed.id}:{item_key(entry)}"


print("Clé item seule :", item_key(entry))
print("Clé composite :", composite_key(test_feed, entry))

Clé item seule : 24ff1a0e-a52a-11f1-85a0-3d03a70b1472
Clé composite : france24:24ff1a0e-a52a-11f1-85a0-3d03a70b1472


## Étape 6 — Premier poll (sans dédup) — debug

On affiche les 3 premiers items de chaque flux. C'est la version « debug » avant `seen.py`.

In [19]:
def poll_all_debug(feeds: list[Feed], limit: int = 3) -> None:
  for feed in feeds:
    parsed = fetch_feed(feed.url)
    print(f"\n=== {feed.id} ({len(parsed.entries)} items) ===")
    for entry in parsed.entries[:limit]:
      print(f"  [{composite_key(feed, entry)}]")
      print(f"    {entry.get('title', '(sans titre)')}")


poll_all_debug(feeds, limit=2)


=== france24 (24 items) ===
  [france24:24ff1a0e-a52a-11f1-85a0-3d03a70b1472]
    Présidentielle 2027 : pourquoi Marine Le Pen bénéficie d'un électorat plus stable que ses rivaux
  [france24:c300791c-a531-11f1-9675-b57e20c7a62b]
    Niger : des arrestations en cours après la mutinerie contre la junte

=== rfi (24 items) ===
  [rfi:71401f6a-a51d-11f1-9cb0-61fb4791cc4c]
    En Suède, Emmanuel Macron doit sceller une vente de frégates pour 4,3 milliards d'euros
  [rfi:b8094756-a521-11f1-b800-a15f442a71c0]
    G20: Washington veut obliger ses alliés à renforcer sa guerre économique contre l'Iran

=== lemonde-une (16 items) ===
  [lemonde-une:https://www.lemonde.fr/politique/article/2026/08/31/retraites-sebastien-lecornu-confirme-la-piste-de-la-desindexation_6762446_823448.html]
    Retraites : Sébastien Lecornu confirme la piste de la désindexation
  [lemonde-une:https://www.lemonde.fr/planete/article/2026/08/31/au-nepal-les-glaciologues-sideres-face-a-l-ampleur-de-la-catastrophe_6762244_

## Étape 7 — `seen.py` : mémoriser ce qu'on a déjà vu

**Rôle** : fichier JSON local `{ "clé": "timestamp_iso" }`.

- 1er poll → beaucoup de clés nouvelles → on les enregistre
- 2e poll immédiat → toutes les clés existent déjà → **0 nouveau**

Ce n'est **pas** encore MinIO (module 03) ni Postgres (module 04).

In [20]:
def load_seen(path: Path = SEEN_PATH) -> dict[str, str]:
  if not path.exists():
    return {}  # premier lancement : rien vu
  return json.loads(path.read_text(encoding="utf-8"))


def save_seen(seen: dict[str, str], path: Path = SEEN_PATH) -> None:
  path.parent.mkdir(parents=True, exist_ok=True)  # crée data/ si absent
  path.write_text(
    json.dumps(seen, indent=2, ensure_ascii=False),
    encoding="utf-8",
  )


def mark_seen(seen: dict[str, str], key: str) -> bool:
  """Retourne True si la clé existait déjà (= doublon)."""
  if key in seen:
    return True
  seen[key] = datetime.now(timezone.utc).isoformat()
  return False


# Démo sur une fausse clé
demo_seen: dict[str, str] = {}
print("1er passage :", mark_seen(demo_seen, "france24:abc"))  # False = nouveau
print("2e passage  :", mark_seen(demo_seen, "france24:abc"))  # True = doublon
print("État seen   :", demo_seen)

1er passage : False
2e passage  : True
État seen   : {'france24:abc': '2026-08-31T13:15:19.051609+00:00'}


## Étape 8 — Poll complet AVEC dédup (cœur du module 02)

Assemble `load_feeds` + `fetch_feed` + `composite_key` + `seen`.

In [21]:
def poll_feed(feed: Feed, seen: dict[str, str]) -> int:
  """Poll un flux. Retourne le nombre de NOUVEAUX items."""
  parsed = fetch_feed(feed.url)
  new_count = 0
  for entry in parsed.entries:
    key = composite_key(feed, entry)
    if mark_seen(seen, key):
      continue  # déjà vu → on saute
    new_count += 1
    print(f"[NEW] {feed.id} | {entry.get('title', '?')}")
  return new_count


def poll_all_dedup(feeds: list[Feed], seen_path: Path = SEEN_PATH) -> int:
  seen = load_seen(seen_path)
  total_new = 0
  for feed in feeds:
    total_new += poll_feed(feed, seen)
  save_seen(seen, seen_path)
  print(f"\n→ {total_new} nouvel(s) item(s) — état sauvé dans {seen_path}")
  return total_new

## Étape 9 — Critère *done* : 1er run vs 2e run

Exécute la cellule **deux fois** d'affilée.

- **1ère fois** : beaucoup de `[NEW]`
- **2ème fois** : `→ 0 nouvel(s) item(s)`

In [22]:
# ⚠️ Pour retester depuis zéro, décommente la ligne suivante :
# SEEN_PATH.unlink(missing_ok=True)

poll_all_dedup(feeds)

[NEW] eu-press | Factsheet: Europe responds, solidarity in action

→ 1 nouvel(s) item(s) — état sauvé dans /home/anthony-marais/Documents/data_project/data/seen.json


1

## Étape 10 — Vérifier `data/seen.json`

Tu dois voir des clés du type `france24:uuid...` avec un timestamp ISO.

In [23]:
if SEEN_PATH.exists():
  seen_data = json.loads(SEEN_PATH.read_text(encoding="utf-8"))
  print(f"{len(seen_data)} clés enregistrées")
  # Affiche 3 exemples
  for i, (k, ts) in enumerate(seen_data.items()):
    if i >= 3:
      break
    print(f"  {k[:60]}... @ {ts}")
else:
  print("seen.json pas encore créé — lance l'étape 9 d'abord")

132 clés enregistrées
  france24:24ff1a0e-a52a-11f1-85a0-3d03a70b1472... @ 2026-08-31T13:12:51.507547+00:00
  france24:c300791c-a531-11f1-9675-b57e20c7a62b... @ 2026-08-31T13:12:51.507769+00:00
  france24:45a4a818-a530-11f1-b8d0-89d95a3cd0c9... @ 2026-08-31T13:12:51.507810+00:00


## Étape 11 — Bonus : headers HTTP (watermarks futurs)

Certains flux (CNIL, HNRSS) envoient `ETag` / `Last-Modified`. Tu pourras les stocker pour éviter de re-télécharger un flux inchangé (304 Not Modified). Pas obligatoire pour le *done* module 02.

In [24]:
cnil = next(f for f in feeds if f.id == "cnil")
resp = httpx.get(cnil.url, headers={"User-Agent": USER_AGENT}, follow_redirects=True)

print("Status :", resp.status_code)
print("ETag           :", resp.headers.get("etag", "(absent)"))
print("Last-Modified  :", resp.headers.get("last-modified", "(absent)"))

Status : 200
ETag           : "1788181935-gzip"
Last-Modified  : Mon, 31 Aug 2026 13:12:15 GMT


## Étape 12 — Comment ça devient `cli.py` ?

Le CLI ne fait qu'**envelopper** ce que tu viens de tester :

```python
# cli.py (schéma)
def main():
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="command", required=True)
    sub.add_parser("poll")
    args = parser.parse_args()
    if args.command == "poll":
        poll_all_dedup(load_feeds())  # ← exactement l'étape 9
```

**Refactorisation** : copie les fonctions validées ici vers :

| Notebook (ici) | Fichier prod |
|---|---|
| `Feed`, `load_feeds` | `src/presslake/ingest/feeds.py` |
| `fetch_feed`, `item_key`, `poll_feed` | `src/presslake/ingest/poll.py` |
| `load_seen`, `save_seen`, `mark_seen` | `src/presslake/ingest/seen.py` |
| `main()` argparse | `src/presslake/cli.py` |

Puis `__init__.py` expose `main` pour `uv run presslake poll`.

## Étape 13 — Bug repéré dans ton `poll.py` actuel

Dans `src/presslake/ingest/poll.py` tu as écrit :

```python
header={"User-Agent": USER_AGENT}  # ❌ header (singulier)
```

httpx attend :

```python
headers={"User-Agent": USER_AGENT}  # ✅ headers (pluriel)
```

Sans ça, le User-Agent n'est pas envoyé → risque de 403.

## Récap — flux de données PressLake (module 02 seulement)

```
feeds.yml → Feed[] → pour chaque URL :
              httpx GET (XML)
              feedparser → entries[]
              pour chaque entry :
                clé = feed_id + item_key
                si clé ∉ seen → [NEW] + sauvegarde
                sinon → skip (dédup)
```

**Module 03** ajoutera : écriture du XML brut dans MinIO bronze.  
**Module 04** ajoutera : catalogue Postgres (`s3_uri`, hash, idempotence).

→ Quand le notebook tourne (2e poll = 0), retourne au tuto [`docs/modules/02-rss-atom.md`](../docs/modules/02-rss-atom.md) section 4h pour finaliser `cli.py`.